In [2]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # para importar desde src/

import pandas as pd
from src.models.dataset import (
    CATEGORICAL_COLS,
    EXCLUDED_COLS,
    fit_categorical_mappings,
    apply_categorical_mappings,
    load_features,
    split_xy,
)

In [3]:
# Load

df = load_features()
print(df.shape)
print(df['split'].value_counts())
df.head(2)


(590540, 460)
split
train    413378
val       88581
test      88581
Name: count, dtype: int64


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,card2_freq,addr1_freq,P_emaildomain_freq,R_emaildomain_freq,DeviceInfo_freq,count_uid2,mean_amt_uid2,std_amt_uid2,amt_ratio_to_uid2_mean,amt_zscore_uid2
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,15959.0,NaN,NaN,NaN,1.0,68.5,NaN,1.0,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,2356.0,29730.0,159214.0,NaN,NaN,1.0,29.0,NaN,1.0,NaN


In [4]:
# Fit de mappings

df_train = df[df['split'] == 'train']
mappings = fit_categorical_mappings(df_train, columns=CATEGORICAL_COLS)

for col, cats in mappings.items():
    print(f"{col:12s} {len(cats):4d} cats: {cats[:5]}{'...' if len(cats) > 5 else ''}")

ProductCD       5 cats: ['C', 'H', 'R', 'S', 'W']
card4           4 cats: ['american express', 'discover', 'mastercard', 'visa']
card6           4 cats: ['charge card', 'credit', 'debit', 'debit or credit']
M1              2 cats: ['F', 'T']
M2              2 cats: ['F', 'T']
M3              2 cats: ['F', 'T']
M4              3 cats: ['M0', 'M1', 'M2']
M5              2 cats: ['F', 'T']
M6              2 cats: ['F', 'T']
M7              2 cats: ['F', 'T']
M8              2 cats: ['F', 'T']
M9              2 cats: ['F', 'T']
DeviceType      2 cats: ['desktop', 'mobile']


In [5]:
df_test_small = df.head(100).copy()
original_dtypes = df_test_small[CATEGORICAL_COLS].dtypes.copy()

df_applied = apply_categorical_mappings(df_test_small, mappings)

# 1. No mutó el original
assert (df_test_small[CATEGORICAL_COLS].dtypes == original_dtypes).all(), "MUTÓ EL ORIGINAL"
print("OK: no mutó el original")

# 2. Las columnas categóricas ahora son Categorical
for col in CATEGORICAL_COLS:
    assert isinstance(df_applied[col].dtype, pd.CategoricalDtype), f"{col} no es Categorical"
print("OK: todas las cols categóricas son Categorical")

# 3. Las categorías matchean el mapping
for col in CATEGORICAL_COLS:
    assert list(df_applied[col].cat.categories) == mappings[col], f"{col} mismatch de categorías"
print("OK: categorías matchean mappings")

OK: no mutó el original
OK: todas las cols categóricas son Categorical
OK: categorías matchean mappings


In [7]:
# Forzá una categoría inventada en la primera fila
df_unseen = df.head(5).copy()
df_unseen.loc[df_unseen.index[0], 'ProductCD'] = 'ZZZ_UNSEEN'

df_unseen_applied = apply_categorical_mappings(df_unseen, mappings)
print(df_unseen_applied['ProductCD'].tolist())
# Esperado: [NaN, valor, valor, valor, valor]
assert pd.isna(df_unseen_applied['ProductCD'].iloc[0]), "Unseen no se convirtió a NaN"
print("OK: unseen → NaN")

[nan, 'W', 'W', 'W', 'H']
OK: unseen → NaN


In [8]:
X_train, y_train, X_val, y_val, X_test, y_test, cat_cols = split_xy(df)

print(f"\nShapes:")
print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  X_val:   {X_val.shape}, y_val:   {y_val.shape}")
print(f"  X_test:  {X_test.shape}, y_test:  {y_test.shape}")

print(f"\nFraud rate por split:")
print(f"  train: {y_train.mean():.4%}")
print(f"  val:   {y_val.mean():.4%}")
print(f"  test:  {y_test.mean():.4%}")

print(f"\nCategorical cols ({len(cat_cols)}): {cat_cols}")

  train: X (413378, 449)     y fraud rate: 3.52%
  val  : X (88581, 449)      y fraud rate: 3.43%
  test : X (88581, 449)      y fraud rate: 3.48%

Shapes:
  X_train: (413378, 449), y_train: (413378,)
  X_val:   (88581, 449), y_val:   (88581,)
  X_test:  (88581, 449), y_test:  (88581,)

Fraud rate por split:
  train: 3.5169%
  val:   3.4341%
  test:  3.4804%

Categorical cols (13): ['ProductCD', 'card4', 'card6', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'DeviceType']


In [9]:
for col in EXCLUDED_COLS:
    assert col not in X_train.columns, f"{col} no fue excluida"
print("OK: todas las EXCLUDED_COLS están fuera de X")

# Y que las categóricas SÍ estén y sean Categorical
for col in CATEGORICAL_COLS:
    assert col in X_train.columns, f"{col} no está en X_train"
    assert isinstance(X_train[col].dtype, pd.CategoricalDtype), f"{col} no es Categorical en X_train"
print("OK: categóricas presentes y tipadas")

OK: todas las EXCLUDED_COLS están fuera de X
OK: categóricas presentes y tipadas


In [10]:
# Las categorías deben ser IDÉNTICAS en train/val/test (esa es la razón de existir del artefacto)
for col in CATEGORICAL_COLS:
    cats_train = list(X_train[col].cat.categories)
    cats_val   = list(X_val[col].cat.categories)
    cats_test  = list(X_test[col].cat.categories)
    assert cats_train == cats_val == cats_test, f"{col}: categorías distintas entre splits"
print("OK: categorías idénticas en train/val/test")

OK: categorías idénticas en train/val/test
